# 03 – Eval & GGUF-Export
Input: Output von 02 (`adapter_dpo`). Vergleicht Basis vs. trainiert auf `eval/eval_prompts.jsonl`.

In [ ]:
# === Setup: Repo klonen + Abhängigkeiten ===
REPO_URL = "https://github.com/Lommel123/creativ-model.git"   # <- anpassen

import os, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass  # öffentliches Repo oder kein Secret gesetzt

if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
%pip install -q unsloth pyyaml
import yaml
CFG = yaml.safe_load(open("repo/config.yaml"))
CFG
%pip install -q sentence-transformers

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

def chat(tokenizer, messages, add_generation_prompt=False):
    kw = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:
        return tokenizer.apply_chat_template(messages, enable_thinking=False, **kw)  # Qwen3: Thinking-Modus aus
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kw)
import json, torch, gc

TUNED = "/kaggle/input/02-dpo/adapter_dpo"   # <- anpassen
prompts = [json.loads(l) for l in open("repo/eval/eval_prompts.jsonl", encoding="utf-8")]
E = CFG["eval"]

def run(model_name, out_path):
    model, tok = FastLanguageModel.from_pretrained(model_name=model_name,
        max_seq_length=CFG["max_seq_length"], load_in_4bit=True)
    FastLanguageModel.for_inference(model)
    with open(out_path, "w", encoding="utf-8") as f:
        for r in prompts:
            p = chat(tok, [{"role":"user","content":r["prompt"]}], add_generation_prompt=True)
            ids = tok(p, return_tensors="pt").to("cuda")
            outs = []
            for _ in range(E["samples_per_prompt"]):
                g = model.generate(**ids, max_new_tokens=350, do_sample=True, temperature=E["temperature"])
                outs.append(tok.decode(g[0][ids.input_ids.shape[1]:], skip_special_tokens=True))
            f.write(json.dumps({**r, "outputs": outs}, ensure_ascii=False) + "\n")
    return model, tok

m, t = run(CFG["base_model"], "out_base.jsonl")
del m; gc.collect(); torch.cuda.empty_cache()
model, tokenizer = run(TUNED, "out_tuned.jsonl")

In [ ]:
!python repo/scripts/score_novelty.py --base out_base.jsonl --tuned out_tuned.jsonl

In [ ]:
# Stichprobe nebeneinander lesen – Gültigkeit prüfst du am besten selbst
base  = [json.loads(l) for l in open("out_base.jsonl")]
tuned = [json.loads(l) for l in open("out_tuned.jsonl")]
for b, u in list(zip(base, tuned))[:5]:
    print("="*80, "\nPROMPT:", b["prompt"])
    print("\n--- BASIS ---\n", b["outputs"][0][:700])
    print("\n--- TRAINIERT ---\n", u["outputs"][0][:700])

In [ ]:
# GGUF für Ollama / LM Studio (q4_k_m ≈ 5 GB für 8B)
model.save_pretrained_gguf("gguf", tokenizer, quantization_method="q4_k_m")
!ls -lh gguf
# Danach 'Save Version' → GGUF unter Output herunterladen